In [ ]:
!pip install catboost

import numpy as np
import tensorflow as tf
from tensorflow.keras.models import load_model
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from xgboost import XGBClassifier
from catboost import CatBoostClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 8.6 MB/s eta 0:00:00


In [ ]:
# Step 1: Load the Base Models (DenseNet169 and Xception for Fruit)
fruit_model_1_path = '/content/drive/MyDrive/Capstone Models/DenseNet169_fruit.keras'
fruit_model_2_path = '/content/drive/MyDrive/Capstone Models/Xception_fruit.keras'

fruit_model_1 = load_model(fruit_model_1_path)
fruit_model_2 = load_model(fruit_model_2_path)

In [ ]:
# Step 2: Load the Dataset (Fruit Test Dataset)
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
SEED = 42

test_dir = "/content/drive/MyDrive/Capstone Dataset/Guava_Fruit_Dieases/Test"

# Define fruit classes manually
fruit_classes = ['Anthracnose', 'Healthy', 'Scab', 'Styler end root']

# Load the test dataset
test_ds = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode="int",
    shuffle=False
)

Found 880 files belonging to 4 classes.


In [ ]:
# Step 3: Generate Predictions from Base Models
y_prob_fruit_model_1 = fruit_model_1.predict(test_ds)  # Predictions from DenseNet169
y_prob_fruit_model_2 = fruit_model_2.predict(test_ds)  # Predictions from Xception

# Step 4: Stack the Predictions (Create Meta-Features)
X_meta_fruit = np.column_stack((
    y_prob_fruit_model_1,
    y_prob_fruit_model_2
))

# True labels for the test set (adjust this to match your dataset)
y_true = np.concatenate([y.numpy() for _, y in test_ds], axis=0)

28/28 ━━━━━━━━━━━━━━━━━━━━ 331s 11s/step
28/28 ━━━━━━━━━━━━━━━━━━━━ 10s 248ms/step


# Step 5: Define and Train Meta-Models (Each Section)

In [ ]:
## 1. Logistic Regression
print("\nTraining Logistic Regression...")
meta_model_lr = LogisticRegression()
meta_model_lr.fit(X_meta_fruit, y_true)
y_meta_pred_lr = meta_model_lr.predict(X_meta_fruit)
print(f"Logistic Regression Accuracy: {accuracy_score(y_true, y_meta_pred_lr):.4f}")
print(classification_report(y_true, y_meta_pred_lr, target_names=fruit_classes))


Training Logistic Regression...
Logistic Regression Accuracy: 0.7693
                 precision    recall  f1-score   support

    Anthracnose       0.78      0.56      0.65       220
        Healthy       0.73      0.93      0.82       220
           Scab       0.72      0.70      0.71       220
Styler end root       0.85      0.89      0.87       220

       accuracy                           0.77       880
      macro avg       0.77      0.77      0.76       880
   weighted avg       0.77      0.77      0.76       880



In [ ]:
## 2. Random Forest
print("\nTraining Random Forest...")
meta_model_rf = RandomForestClassifier()
meta_model_rf.fit(X_meta_fruit, y_true)
y_meta_pred_rf = meta_model_rf.predict(X_meta_fruit)
print(f"Random Forest Accuracy: {accuracy_score(y_true, y_meta_pred_rf):.4f}")
print(classification_report(y_true, y_meta_pred_rf, target_names=fruit_classes))


Training Random Forest...
Random Forest Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 3. Gradient Boosting
print("\nTraining Gradient Boosting...")
meta_model_gb = GradientBoostingClassifier()
meta_model_gb.fit(X_meta_fruit, y_true)
y_meta_pred_gb = meta_model_gb.predict(X_meta_fruit)
print(f"Gradient Boosting Accuracy: {accuracy_score(y_true, y_meta_pred_gb):.4f}")
print(classification_report(y_true, y_meta_pred_gb, target_names=fruit_classes))


Training Gradient Boosting...
Gradient Boosting Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 4. Support Vector Classifier (SVC)
print("\nTraining Support Vector Classifier (SVC)...")
meta_model_svc = SVC()
meta_model_svc.fit(X_meta_fruit, y_true)
y_meta_pred_svc = meta_model_svc.predict(X_meta_fruit)
print(f"SVC Accuracy: {accuracy_score(y_true, y_meta_pred_svc):.4f}")
print(classification_report(y_true, y_meta_pred_svc, target_names=fruit_classes))


Training Support Vector Classifier (SVC)...
SVC Accuracy: 0.8057
                 precision    recall  f1-score   support

    Anthracnose       0.84      0.64      0.73       220
        Healthy       0.77      0.91      0.84       220
           Scab       0.71      0.79      0.75       220
Styler end root       0.92      0.88      0.90       220

       accuracy                           0.81       880
      macro avg       0.81      0.81      0.80       880
   weighted avg       0.81      0.81      0.80       880



In [ ]:
## 5. Multi-layer Perceptron (MLP)
print("\nTraining Multi-layer Perceptron (MLP)...")
meta_model_mlp = MLPClassifier()
meta_model_mlp.fit(X_meta_fruit, y_true)
y_meta_pred_mlp = meta_model_mlp.predict(X_meta_fruit)
print(f"MLP Accuracy: {accuracy_score(y_true, y_meta_pred_mlp):.4f}")
print(classification_report(y_true, y_meta_pred_mlp, target_names=fruit_classes))


Training Multi-layer Perceptron (MLP)...
MLP Accuracy: 0.8148
                 precision    recall  f1-score   support

    Anthracnose       0.83      0.71      0.76       220
        Healthy       0.79      0.91      0.84       220
           Scab       0.76      0.75      0.75       220
Styler end root       0.88      0.90      0.89       220

       accuracy                           0.81       880
      macro avg       0.82      0.81      0.81       880
   weighted avg       0.82      0.81      0.81       880



/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (200) reached and the optimization hasn't converged yet.
  warnings.warn(


In [ ]:
## 6. AdaBoost Classifier
print("\nTraining AdaBoost Classifier...")
meta_model_ada = AdaBoostClassifier()
meta_model_ada.fit(X_meta_fruit, y_true)
y_meta_pred_ada = meta_model_ada.predict(X_meta_fruit)
print(f"AdaBoost Accuracy: {accuracy_score(y_true, y_meta_pred_ada):.4f}")
print(classification_report(y_true, y_meta_pred_ada, target_names=fruit_classes))


Training AdaBoost Classifier...
AdaBoost Accuracy: 0.7955
                 precision    recall  f1-score   support

    Anthracnose       0.77      0.70      0.73       220
        Healthy       0.78      0.91      0.84       220
           Scab       0.82      0.66      0.73       220
Styler end root       0.82      0.90      0.86       220

       accuracy                           0.80       880
      macro avg       0.80      0.80      0.79       880
   weighted avg       0.80      0.80      0.79       880



In [ ]:
## 7. Decision Tree Classifier
print("\nTraining Decision Tree Classifier...")
meta_model_dt = DecisionTreeClassifier()
meta_model_dt.fit(X_meta_fruit, y_true)
y_meta_pred_dt = meta_model_dt.predict(X_meta_fruit)
print(f"Decision Tree Accuracy: {accuracy_score(y_true, y_meta_pred_dt):.4f}")
print(classification_report(y_true, y_meta_pred_dt, target_names=fruit_classes))


Training Decision Tree Classifier...
Decision Tree Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 8. K-Nearest Neighbors (KNN)
print("\nTraining K-Nearest Neighbors (KNN)...")
meta_model_knn = KNeighborsClassifier()
meta_model_knn.fit(X_meta_fruit, y_true)
y_meta_pred_knn = meta_model_knn.predict(X_meta_fruit)
print(f"KNN Accuracy: {accuracy_score(y_true, y_meta_pred_knn):.4f}")
print(classification_report(y_true, y_meta_pred_knn, target_names=fruit_classes))


Training K-Nearest Neighbors (KNN)...
KNN Accuracy: 0.9148
                 precision    recall  f1-score   support

    Anthracnose       0.92      0.89      0.91       220
        Healthy       0.92      0.93      0.93       220
           Scab       0.87      0.87      0.87       220
Styler end root       0.94      0.97      0.96       220

       accuracy                           0.91       880
      macro avg       0.91      0.91      0.91       880
   weighted avg       0.91      0.91      0.91       880



In [ ]:
## 9. Gaussian Naive Bayes (GNB)
print("\nTraining Gaussian Naive Bayes (GNB)...")
meta_model_gnb = GaussianNB()
meta_model_gnb.fit(X_meta_fruit, y_true)
y_meta_pred_gnb = meta_model_gnb.predict(X_meta_fruit)
print(f"GNB Accuracy: {accuracy_score(y_true, y_meta_pred_gnb):.4f}")
print(classification_report(y_true, y_meta_pred_gnb, target_names=fruit_classes))


Training Gaussian Naive Bayes (GNB)...
GNB Accuracy: 0.7409
                 precision    recall  f1-score   support

    Anthracnose       0.74      0.46      0.57       220
        Healthy       0.73      0.90      0.81       220
           Scab       0.71      0.68      0.69       220
Styler end root       0.78      0.92      0.85       220

       accuracy                           0.74       880
      macro avg       0.74      0.74      0.73       880
   weighted avg       0.74      0.74      0.73       880



In [ ]:
## 10. Quadratic Discriminant Analysis (QDA)
print("\nTraining Quadratic Discriminant Analysis (QDA)...")
meta_model_qda = QuadraticDiscriminantAnalysis()
meta_model_qda.fit(X_meta_fruit, y_true)
y_meta_pred_qda = meta_model_qda.predict(X_meta_fruit)
print(f"QDA Accuracy: {accuracy_score(y_true, y_meta_pred_qda):.4f}")
print(classification_report(y_true, y_meta_pred_qda, target_names=fruit_classes))


Training Quadratic Discriminant Analysis (QDA)...
QDA Accuracy: 0.7443
                 precision    recall  f1-score   support

    Anthracnose       0.75      0.44      0.55       220
        Healthy       0.77      0.92      0.84       220
           Scab       0.68      0.68      0.68       220
Styler end root       0.77      0.93      0.84       220

       accuracy                           0.74       880
      macro avg       0.74      0.74      0.73       880
   weighted avg       0.74      0.74      0.73       880



/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 0 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 1 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 2 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/discriminant_analysis.py:1024: LinAlgWarning: The covariance matrix of class 3 is not full rank. Increasing the value of parameter `reg_param` might help reducing the collinearity.
  warnings.warn(


In [ ]:
## 11. XGBoost Classifier
print("\nTraining XGBoost Classifier...")
meta_model_xgb = XGBClassifier()
meta_model_xgb.fit(X_meta_fruit, y_true)
y_meta_pred_xgb = meta_model_xgb.predict(X_meta_fruit)
print(f"XGBoost Accuracy: {accuracy_score(y_true, y_meta_pred_xgb):.4f}")
print(classification_report(y_true, y_meta_pred_xgb, target_names=fruit_classes))


Training XGBoost Classifier...
XGBoost Accuracy: 1.0000
                 precision    recall  f1-score   support

    Anthracnose       1.00      1.00      1.00       220
        Healthy       1.00      1.00      1.00       220
           Scab       1.00      1.00      1.00       220
Styler end root       1.00      1.00      1.00       220

       accuracy                           1.00       880
      macro avg       1.00      1.00      1.00       880
   weighted avg       1.00      1.00      1.00       880



In [ ]:
## 12. CatBoost Classifier
print("\nTraining CatBoost Classifier...")
meta_model_cat = CatBoostClassifier(learning_rate=0.1, iterations=1000, depth=6)
meta_model_cat.fit(X_meta_fruit, y_true)
y_meta_pred_cat = meta_model_cat.predict(X_meta_fruit)
print(f"CatBoost Accuracy: {accuracy_score(y_true, y_meta_pred_cat):.4f}")
print(classification_report(y_true, y_meta_pred_cat, target_names=fruit_classes))


Training CatBoost Classifier...
0:	learn: 1.2561857	total: 53.6ms	remaining: 53.6s
1:	learn: 1.1578383	total: 58.8ms	remaining: 29.3s
2:	learn: 1.0747073	total: 63.6ms	remaining: 21.1s
3:	learn: 0.9939388	total: 68.7ms	remaining: 17.1s
4:	learn: 0.9324873	total: 73.8ms	remaining: 14.7s
5:	learn: 0.8782491	total: 78.7ms	remaining: 13s
6:	learn: 0.8255860	total: 83.7ms	remaining: 11.9s
7:	learn: 0.7864214	total: 88.7ms	remaining: 11s
8:	learn: 0.7456176	total: 93.9ms	remaining: 10.3s
9:	learn: 0.7114518	total: 98.9ms	remaining: 9.79s
10:	learn: 0.6772556	total: 104ms	remaining: 9.34s
11:	learn: 0.6445177	total: 109ms	remaining: 8.97s
12:	learn: 0.6179753	total: 114ms	remaining: 8.66s
13:	learn: 0.5931557	total: 119ms	remaining: 8.38s
14:	learn: 0.5729915	total: 124ms	remaining: 8.16s
15:	learn: 0.5550025	total: 129ms	remaining: 7.95s
16:	learn: 0.5369564	total: 134ms	remaining: 7.77s
17:	learn: 0.5193280	total: 139ms	remaining: 7.61s
18:	learn: 0.5042429	total: 145ms	remaining: 7.46s
19

In [ ]:
## 13. LightGBM Classifier
print("\nTraining LightGBM Classifier...")
meta_model_lgbm = LGBMClassifier()
meta_model_lgbm.fit(X_meta_fruit, y_true)
y_meta_pred_lgbm = meta_model_lgbm.predict(X_meta_fruit)
print(f"LightGBM Accuracy: {accuracy_score(y_true, y_meta_pred_lgbm):.4f}")
print(classification_report(y_true, y_meta_pred_lgbm, target_names=fruit_classes))


Training LightGBM Classifier...
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000209 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 2040
[LightGBM] [Info] Number of data points in the train set: 880, number of used features: 8
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Info] Start training from score -1.386294
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [War

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
